# SurfaceTarget-CRC — phase 04 — patch 4 1b

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.1B
#
# DEPMAP CRISPR FILE FRAGMENT RECOVERY
#
# Purpose:
# 1. Locate both preserved upload fragments
# 2. Validate their combined expected file size
# 3. Inspect their joining boundary
# 4. Construct a recovery candidate
# 5. Validate the complete CSV structure
# 6. Verify model identifiers against Model.csv
# 7. Preserve the original fragments
# 8. Save recovery provenance and checksums
#
# IMPORTANT:
# - Original fragments are preserved
# - No dependency calculations performed
# - No source measurements intentionally modified
# - Official source checksum comparison remains pending
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import hashlib
import shutil
import csv
import re
import os
import json
import numpy as np


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

DEPMAP = ROOT / "data/raw/depmap/26Q3"
META = ROOT / "data/metadata"

META.mkdir(parents=True, exist_ok=True)

FIRST_FRAGMENT = (
    DEPMAP /
    "CRISPRGeneEffect_incomplete_20261121_081207.csv"
)

SECOND_FRAGMENT = (
    DEPMAP /
    "CRISPRGeneEffect.csv"
)

RECOVERY_CANDIDATE = (
    DEPMAP /
    "CRISPRGeneEffect_recovery_candidate.csv"
)

MODEL_PATH = DEPMAP / "Model.csv"


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.1B")
print("DEPMAP CRISPR FRAGMENT RECOVERY")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE INPUT FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PRESERVED FILE FRAGMENTS")
print("-" * 65)


for path in [
    FIRST_FRAGMENT,
    SECOND_FRAGMENT,
    MODEL_PATH
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing: {path}"
        )


first_size = FIRST_FRAGMENT.stat().st_size

second_size = SECOND_FRAGMENT.stat().st_size

combined_size = first_size + second_size


print("\nFirst fragment:")

print(
    f"Size: {first_size / 1024**2:.2f} MiB"
)


print("\nSecond fragment:")

print(
    f"Size: {second_size / 1024**2:.2f} MiB"
)


print("\nCombined:")

print(
    f"Size: {combined_size / 1024**2:.2f} MiB"
)

print(
    f"Size: {combined_size / 1_000_000:.2f} MB"
)


# The website displayed approximately 497.1 MB.
# This is a preliminary plausibility check,
# not an official file-integrity verification.

if not (490_000_000 <= combined_size <= 505_000_000):

    raise RuntimeError(
        "Combined fragment size is inconsistent "
        "with the expected DepMap download."
    )


print("\nCombined file-size plausibility: PASSED")


# ------------------------------------------------------------
# 3. INSPECT THE FRAGMENT BOUNDARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INSPECTING FRAGMENT BOUNDARY")
print("-" * 65)


with open(FIRST_FRAGMENT, "rb") as f:

    f.seek(max(0, first_size - 100))

    first_tail = f.read()


with open(SECOND_FRAGMENT, "rb") as f:

    second_head = f.read(100)


print("\nEnd of first fragment:")

print(repr(first_tail.decode("utf-8", errors="replace")))


print("\nBeginning of second fragment:")

print(repr(second_head.decode("utf-8", errors="replace")))


print("\nCombined boundary preview:")

boundary = first_tail + second_head

print(repr(boundary.decode("utf-8", errors="replace")))


# ------------------------------------------------------------
# 4. VALIDATE ORIGINAL CSV HEADER
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING ORIGINAL CRISPR HEADER")
print("-" * 65)


with open(
    FIRST_FRAGMENT,
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    reader = csv.reader(f)

    original_header = next(reader)


expected_columns = len(original_header)


print("\nOriginal header columns:", expected_columns)

print("First five headers:")

print(original_header[:5])


if expected_columns < 18000:

    raise RuntimeError(
        "The first fragment does not contain "
        "the expected DepMap gene header."
    )


if original_header[0] not in ["", "ModelID"]:

    raise RuntimeError(
        "Unexpected first column in original header."
    )


# Check the standard gene naming format:
# GeneSymbol (EntrezGeneID)

gene_header_pattern = re.compile(
    r".+\s\(\d+\)$"
)


matching_gene_headers = sum(
    bool(gene_header_pattern.fullmatch(name))
    for name in original_header[1:]
)


header_match_fraction = (
    matching_gene_headers / (expected_columns - 1)
)


print(
    "Gene columns matching expected format:",
    matching_gene_headers
)

print(
    "Matching header fraction:",
    round(header_match_fraction, 5)
)


if header_match_fraction < 0.98:

    raise RuntimeError(
        "Unexpected CRISPR gene-column naming structure."
    )


print("\nOriginal gene header: VALIDATED")


# ------------------------------------------------------------
# 5. LOAD MODEL IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING MODEL METADATA")
print("-" * 65)


models = pd.read_csv(
    MODEL_PATH,
    usecols=["ModelID"],
    dtype=str
)


if models["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate ModelIDs in Model.csv."
    )


model_ids = set(
    models["ModelID"]
    .dropna()
    .astype(str)
    .str.strip()
)


print("\nMetadata models:", len(model_ids))


# ------------------------------------------------------------
# 6. RECONSTRUCT CSV WITHOUT MODIFYING FRAGMENTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RECONSTRUCTING CRISPR CSV")
print("-" * 65)


if RECOVERY_CANDIDATE.exists():

    raise RuntimeError(
        "A previous recovery candidate already exists. "
        "Inspect or remove that candidate before rerunning."
    )


# Copy the original byte sequences in order.
#
# Fragment 1 contains the original header.
# Fragment 2 appears to continue its data.
#
# The next section validates the resulting CSV
# before it can replace the working file.

with open(RECOVERY_CANDIDATE, "wb") as output:

    for fragment in [
        FIRST_FRAGMENT,
        SECOND_FRAGMENT
    ]:

        with open(fragment, "rb") as source:

            shutil.copyfileobj(
                source,
                output,
                length=1024 * 1024
            )


reconstructed_size = (
    RECOVERY_CANDIDATE.stat().st_size
)


if reconstructed_size != combined_size:

    raise RuntimeError(
        "Reconstructed file size does not "
        "match the sum of both fragments."
    )


print("\nRecovery candidate created.")

print(
    "Size:",
    round(reconstructed_size / 1024**2, 2),
    "MiB"
)


# ------------------------------------------------------------
# 7. VALIDATE EVERY RECONSTRUCTED CSV ROW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RECONSTRUCTED CSV STRUCTURE")
print("-" * 65)


observed_ids = set()

row_count = 0

numeric_checks = 0

model_matching_count = 0


# Representative fields across each row.
# The full file is streamed rather than
# materialized as a huge DataFrame.

sample_column_positions = [
    1,
    100,
    1000,
    5000,
    10000,
    expected_columns - 1
]


try:

    with open(
        RECOVERY_CANDIDATE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        reader = csv.reader(
            f,
            strict=True
        )

        reconstructed_header = next(reader)

        if reconstructed_header != original_header:

            raise RuntimeError(
                "Reconstructed CSV header differs "
                "from the preserved original header."
            )

        for row in reader:

            row_count += 1

            # ------------------------------------
            # Check complete CSV row width
            # ------------------------------------

            if len(row) != expected_columns:

                raise RuntimeError(
                    "Malformed CSV row detected. "
                    f"Row: {row_count}, "
                    f"Expected fields: {expected_columns}, "
                    f"Observed fields: {len(row)}"
                )

            # ------------------------------------
            # Validate ModelID
            # ------------------------------------

            model_id = str(row[0]).strip()

            if not re.fullmatch(
                r"ACH-\d+",
                model_id
            ):

                raise RuntimeError(
                    "Unexpected CRISPR ModelID. "
                    f"Row: {row_count}, "
                    f"Observed: {model_id}"
                )

            if model_id in observed_ids:

                raise RuntimeError(
                    f"Duplicate CRISPR ModelID: {model_id}"
                )

            observed_ids.add(model_id)

            if model_id not in model_ids:

                raise RuntimeError(
                    "CRISPR ModelID not present in Model.csv: "
                    f"{model_id}"
                )

            model_matching_count += 1

            # ------------------------------------
            # Check representative gene values
            # ------------------------------------

            for position in sample_column_positions:

                value = row[position].strip()

                if value == "" or value.lower() in {
                    "na", "nan"
                }:

                    continue

                try:

                    numeric_value = float(value)

                except ValueError:

                    raise RuntimeError(
                        "Invalid numerical gene-effect value. "
                        f"Row: {row_count}, "
                        f"Column: {position}, "
                        f"Value: {value}"
                    )

                if not np.isfinite(numeric_value):

                    raise RuntimeError(
                        "Non-finite gene-effect value detected."
                    )

                numeric_checks += 1

            if row_count % 250 == 0:

                print(
                    f"Validated {row_count} CRISPR model rows"
                )

except Exception:

    print(
        "\nRECOVERY VALIDATION FAILED."
    )

    print(
        "Both original fragments remain preserved."
    )

    print(
        "No source measurements have been replaced."
    )

    raise


# ------------------------------------------------------------
# 8. COMPLETE STRUCTURE VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RECONSTRUCTION VALIDATION SUMMARY")
print("-" * 65)


print("\nGene columns:", expected_columns - 1)

print("CRISPR model rows:", row_count)

print("Unique ModelIDs:", len(observed_ids))

print("Models matching metadata:", model_matching_count)

print("Numerical spot-checks:", numeric_checks)


if row_count < 500:

    raise RuntimeError(
        "Unexpectedly few CRISPR model rows."
    )


if row_count > len(model_ids):

    raise RuntimeError(
        "CRISPR model count exceeds metadata models."
    )


if len(observed_ids) != row_count:

    raise RuntimeError(
        "CRISPR model identifier uniqueness failed."
    )


if model_matching_count != row_count:

    raise RuntimeError(
        "Not all CRISPR models matched metadata."
    )


print("\nCSV structure validation: PASSED")

print("Model correspondence: PASSED")

print("Numerical spot-checks: PASSED")


# ------------------------------------------------------------
# 9. CALCULATE SOURCE HASHES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING RECONSTRUCTION CHECKSUMS")
print("-" * 65)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


first_hash = sha256_file(FIRST_FRAGMENT)

second_hash = sha256_file(SECOND_FRAGMENT)

reconstructed_hash = sha256_file(
    RECOVERY_CANDIDATE
)


print("\nFirst fragment SHA256:", first_hash)

print("Second fragment SHA256:", second_hash)

print("Reconstructed SHA256:", reconstructed_hash)


# ------------------------------------------------------------
# 10. PRESERVE FRAGMENTS AND INSTALL VERIFIED CSV
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INSTALLING STRUCTURALLY VALIDATED CSV")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")


tail_backup = (
    DEPMAP /
    f"CRISPRGeneEffect_tail_fragment_{timestamp}.csv"
)


# Preserve the second fragment separately.
# The first fragment is already preserved.

os.replace(
    SECOND_FRAGMENT,
    tail_backup
)


try:

    os.replace(
        RECOVERY_CANDIDATE,
        SECOND_FRAGMENT
    )

except Exception:

    # Restore original working file if needed.

    if (
        tail_backup.exists()
        and not SECOND_FRAGMENT.exists()
    ):

        os.replace(
            tail_backup,
            SECOND_FRAGMENT
        )

    raise


print("\nRecovered CSV installed successfully.")

print("Original first fragment: PRESERVED")

print("Original second fragment: PRESERVED")

print("Recovered full CSV: INSTALLED")


# ------------------------------------------------------------
# 11. SAVE IMPORT AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RECOVERY PROVENANCE")
print("-" * 65)


records = [

    {
        "release": "DepMap Public 26Q3",
        "filename": "Model.csv",
        "local_path": str(MODEL_PATH),
        "file_size_bytes": MODEL_PATH.stat().st_size,
        "sha256": sha256_file(MODEL_PATH),
        "header_valid": True,
        "source": "Official DepMap browser download"
    },

    {
        "release": "DepMap Public 26Q3",
        "filename": "CRISPRGeneEffect.csv",
        "local_path": str(SECOND_FRAGMENT),
        "file_size_bytes":
            SECOND_FRAGMENT.stat().st_size,
        "sha256": reconstructed_hash,
        "header_valid": True,
        "source": (
            "Reconstructed from two preserved "
            "Colab upload fragments"
        )
    }
]


pd.DataFrame(records).to_csv(
    META / "depmap_26q3_import_verification.csv",
    index=False
)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.1B",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "declared_release": "DepMap Public 26Q3",

    "reconstruction_method": (
        "Bytewise concatenation of preserved "
        "first and second upload fragments"
    ),

    "first_fragment_path": str(FIRST_FRAGMENT),

    "second_fragment_backup_path": str(tail_backup),

    "first_fragment_bytes": first_size,

    "second_fragment_bytes": second_size,

    "reconstructed_bytes": reconstructed_size,

    "first_fragment_sha256": first_hash,

    "second_fragment_sha256": second_hash,

    "reconstructed_sha256": reconstructed_hash,

    "crispr_model_rows": row_count,

    "crispr_gene_columns": expected_columns - 1,

    "model_metadata_rows": len(model_ids),

    "models_matching_metadata": model_matching_count,

    "csv_structure_validation": True,

    "numerical_spot_checks": numeric_checks,

    "official_release_checksum_verified": False,

    "important_limitation": (
        "Structural and size validation support "
        "the reconstruction, but do not independently "
        "prove byte-for-byte identity with the "
        "official DepMap download."
    ),

    "dependency_analysis_performed": False,

    "candidate_exclusions": 0
}


with open(
    META / "depmap_26q3_import_provenance.json",
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


print("\nImport audit: SAVED")

print("Reconstruction provenance: SAVED")


# ------------------------------------------------------------
# 12. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.1B SUMMARY")
print("=" * 65)


print("\nDepMap release: 26Q3")

print(
    "Recovered CSV size:",
    round(reconstructed_size / 1024**2, 2),
    "MiB"
)

print("CRISPR model rows:", row_count)

print("CRISPR gene columns:", expected_columns - 1)

print(
    "Model metadata correspondence:",
    "PASSED"
)

print("\nStructural CSV validation: PASSED")

print("Numerical spot-checks: PASSED")

print("Both upload fragments: PRESERVED")

print("Source checksums: SAVED")

print("\nOfficial source checksum verified: NO")

print("CRISPR dependency analysis performed: NO")

print("\nPATCH 4.1B COMPLETE")

print("=" * 65)